In [ ]:
# ============================================================
# EXPERIMENT 02
# Visualization of Convolutional Filters and Feature Maps
# using a Pre-trained VGG16 CNN
# ============================================================


# ============================================================
# 1. IMPORT REQUIRED LIBRARIES
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import tensorflow_datasets as tfds

from tensorflow.keras.applications import VGG16
from tensorflow.keras.applications.vgg16 import preprocess_input
from tensorflow.keras.utils import img_to_array


# ============================================================
# 2. LOAD PRE-TRAINED VGG16 MODEL
# ============================================================

# Load VGG16 with weights already trained on ImageNet.
#
# include_top=True means the complete VGG16 is loaded.
# We need the complete network because we want to
# examine its convolutional layers.

model = VGG16(
    weights="imagenet"
)

print("VGG16 model loaded successfully!")


# ============================================================
# 3. LOAD INPUT IMAGE FROM DATASET
# ============================================================

# Load an ImageNet-derived dataset.
#
# Images are loaded directly as TensorFlow tensors.
# No image files such as cat.jpg or dog.jpg are required.

dataset, info = tfds.load(
    "imagenette/320px-v2",
    split="train",
    as_supervised=True,
    with_info=True,
    shuffle_files=True
)

print("Dataset loaded successfully!")


# Randomly select one image from the dataset.

image, label = next(
    iter(
        dataset.shuffle(
            buffer_size=1000,
            seed=42
        ).take(1)
    )
)


# VGG16 expects an input size of 224 x 224.

image = tf.image.resize(
    image,
    (224, 224)
)


# ============================================================
# 4. DISPLAY ORIGINAL IMAGE
# ============================================================

plt.figure(figsize=(5, 5))

plt.imshow(
    image.numpy().astype("uint8")
)

plt.title("Original Input Image")

plt.axis("off")

plt.show()


# ============================================================
# 5. PREPROCESS THE IMAGE
# ============================================================

# Convert image to NumPy array.

image_array = img_to_array(
    image
).copy()

print("Original image shape:",
      image_array.shape)


# Add batch dimension.
#
# Before:
# (224, 224, 3)
#
# After:
# (1, 224, 224, 3)

image_batch = np.expand_dims(
    image_array,
    axis=0
)

print("Batch shape:",
      image_batch.shape)


# Apply VGG16-specific preprocessing.

processed_image = preprocess_input(
    image_batch
)


# ============================================================
# 6. SELECT THREE CONVOLUTIONAL LAYERS
# ============================================================

# We select layers from different depths.
#
# Early:
# block1_conv1
#
# Intermediate:
# block3_conv3
#
# Deep:
# block5_conv3

layer_names = [
    "block1_conv1",
    "block3_conv3",
    "block5_conv3"
]


# ============================================================
# 7. CREATE A MODEL FOR FEATURE EXTRACTION
# ============================================================

# Instead of getting only the final classification,
# we create a new model that gives us the outputs
# of the selected convolutional layers.

layer_outputs = [
    model.get_layer(name).output
    for name in layer_names
]


feature_model = tf.keras.Model(
    inputs=model.input,
    outputs=layer_outputs
)


# ============================================================
# 8. EXTRACT FEATURE MAPS
# ============================================================

# Pass the image through VGG16.
#
# The output will contain the feature maps from
# all three selected layers.

feature_maps = feature_model.predict(
    processed_image,
    verbose=0
)


# ============================================================
# 9. DISPLAY FEATURE MAP DIMENSIONS
# ============================================================

print("\n============================================")
print("FEATURE MAP DIMENSIONS")
print("============================================")

for name, feature_map in zip(
        layer_names,
        feature_maps):

    print(
        name,
        "->",
        feature_map.shape
    )


# ============================================================
# 10. VISUALIZE FEATURE MAPS
# ============================================================

# Number of feature maps to display from each layer.

num_maps = 6


for layer_name, feature_map in zip(
        layer_names,
        feature_maps):


    # --------------------------------------------------------
    # Remove batch dimension
    # --------------------------------------------------------

    # Example:
    #
    # Before:
    # (1, height, width, channels)
    #
    # After:
    # (height, width, channels)

    feature_map = feature_map[0]


    # --------------------------------------------------------
    # Display feature maps
    # --------------------------------------------------------

    plt.figure(
        figsize=(12, 6)
    )

    for i in range(num_maps):

        plt.subplot(2, 3, i + 1)

        plt.imshow(
            feature_map[:, :, i],
            cmap="viridis"
        )

        plt.title(
            "Feature Map " + str(i + 1)
        )

        plt.axis("off")


    plt.suptitle(
        "Feature Maps - " + layer_name
    )

    plt.tight_layout()

    plt.show()


# ============================================================
# 11. VISUALIZE LEARNED CONVOLUTIONAL FILTERS
# ============================================================

# We will visualize the filters from the selected
# convolutional layers.
#
# A convolutional layer stores filters in its weights.

for layer_name in layer_names:

    # Get the selected layer.
    layer = model.get_layer(layer_name)


    # Get the layer weights.
    weights = layer.get_weights()


    # The first element contains the convolution filters.
    filters = weights[0]


    print("\n============================================")
    print("FILTER INFORMATION")
    print("============================================")

    print("Layer:", layer_name)

    print("Filter shape:",
          filters.shape)


    # --------------------------------------------------------
    # Display the first few filters
    # --------------------------------------------------------

    plt.figure(
        figsize=(12, 6)
    )


    # Display up to 6 filters.

    num_filters = min(6, filters.shape[3])


    for i in range(num_filters):

        plt.subplot(2, 3, i + 1)


        # Get one filter.
        filter_image = filters[:, :, :, i]


        # Normalize values between 0 and 1
        # so that the filter can be displayed.

        filter_min = filter_image.min()
        filter_max = filter_image.max()

        filter_image = (
            filter_image - filter_min
        ) / (
            filter_max - filter_min + 1e-8
        )


        # Display filter.
        #
        # Some filters have 3 input channels,
        # so they can be displayed as RGB images.

        if filter_image.shape[2] == 3:

            plt.imshow(
                filter_image
            )

        else:

            plt.imshow(
                filter_image[:, :, 0],
                cmap="viridis"
            )


        plt.title(
            "Filter " + str(i + 1)
        )

        plt.axis("off")


    plt.suptitle(
        "Learned Filters - " + layer_name
    )

    plt.tight_layout()

    plt.show()


# ============================================================
# 12. PRINT FINAL COMPARISON
# ============================================================

print("\n============================================")
print("LAYER COMPARISON")
print("============================================")

print("Early Layer       : block1_conv1")
print("Intermediate Layer: block3_conv3")
print("Deep Layer        : block5_conv3")

print("\nAs depth increases:")
print("1. Spatial dimensions become smaller.")
print("2. Number of channels increases.")
print("3. Features become more complex.")
print("4. Early layers detect edges and simple patterns.")
print("5. Intermediate layers detect textures and shapes.")
print("6. Deep layers detect high-level object features.")

print("\nExperiment completed successfully!")